In [ ]:
import json
from collections import Counter

with open('/kaggle/input/datasets/biancaroman/adnotari-pmb/project-1-at-2026-04-02-16-43-5ed6b2eb.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total texte: {len(data)}")

lc = Counter()
texte_cu_adnotari = 0
for task in data:
    anns = task.get('annotations', [])
    if anns and anns[0].get('result'):
        texte_cu_adnotari += 1
        for r in anns[0]['result']:
            lc[r['value']['labels'][0]] += 1

print(f"Texte cu adnotari: {texte_cu_adnotari}")
print(f"\nDistributie etichete:")
for l, c in lc.most_common():
    print(f"  {l}: {c}")

In [ ]:
!pip install -q datasets seqeval transformers torch

In [ ]:
import json, re
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset
from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1
from collections import Counter

In [ ]:
with open('/kaggle/input/datasets/biancaroman/adnotari-pmb/project-1-at-2026-04-02-16-43-5ed6b2eb.json',
          'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total texte: {len(data)}")

# scot cnp din adnotari
for task in data:
    anns = task.get('annotations', [])
    if anns and anns[0].get('result'):
        anns[0]['result'] = [
            r for r in anns[0]['result']
            if r['value']['labels'][0] != 'CNP'
        ]

lc = Counter()
for task in data:
    anns = task.get('annotations', [])
    if anns and anns[0].get('result'):
        for r in anns[0]['result']:
            lc[r['value']['labels'][0]] += 1
print(f"Distributie dupa scoatere CNP:")
for l, c in lc.most_common():
    print(f"  {l}: {c}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('Davlan/xlm-roberta-base-ner-hrl')
label_list = ['O', 'B-PER_ANON', 'I-PER_ANON',
              'B-ADDR_DOM', 'I-ADDR_DOM',
              'B-DOC_ID',   'I-DOC_ID']
label2id = {l: i for i, l in enumerate(label_list)}
id2label  = {i: l for i, l in enumerate(label_list)}

In [ ]:
def convert_to_iob(task, tokenizer, max_length=512):
    text = task['data']['text']

    annotations = []
    anns = task.get('annotations', [])
    if anns and anns[0].get('result'):
        for r in anns[0]['result']:
            annotations.append({
                'start': r['value']['start'],
                'end':   r['value']['end'],
                'label': r['value']['labels'][0]
            })
    annotations.sort(key=lambda x: x['start'])

    encoding = tokenizer(text, truncation=True, max_length=max_length,
                         return_offsets_mapping=True, return_tensors=None)
    offsets = encoding['offset_mapping']
    labels  = ['O'] * len(encoding['input_ids'])

    for ann in annotations:
        first_token = True
        for idx, (ts, te) in enumerate(offsets):
            if te == 0:  
                continue
            if (ts >= ann['start'] and te <= ann['end']) or \
               (ts < ann['end'] and te > ann['start']):
                labels[idx] = f"B-{ann['label']}" if first_token else f"I-{ann['label']}"
                first_token = False

    return {
        'input_ids':      encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels':         [label2id[l] for l in labels],
    }

converted = [convert_to_iob(task, tokenizer) for task in data]
print(f"Convertite: {len(converted)}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/biancaroman/pmb-dosare2/pmb_dosare2.csv')

dosar_map = {}
for i, task in enumerate(data):
    fname = task['data'].get('filename', str(i))
    dosar_nr = fname.split('_')[0]
    if dosar_nr not in dosar_map:
        dosar_map[dosar_nr] = []
    dosar_map[dosar_nr].append(i)

df['dosar_nr'] = df['Dosar PMB'].str.split('/').str[0].str.strip()

dosar_to_solicitant = df.drop_duplicates('dosar_nr').set_index('dosar_nr')['Solicitant'].to_dict()

from collections import defaultdict
solicitant_to_dosare = defaultdict(list)
for dosar_nr in dosar_map.keys():
    solicitant = dosar_to_solicitant.get(dosar_nr, dosar_nr)  # fallback la dosar_nr
    solicitant_to_dosare[solicitant].append(dosar_nr)

# split pe solicitant
solicitanti = list(solicitant_to_dosare.keys())
train_sol, test_sol = train_test_split(solicitanti, test_size=0.28, random_state=42)

train_idx = [i for s in train_sol for d in solicitant_to_dosare[s] for i in dosar_map[d]]
test_idx  = [i for s in test_sol  for d in solicitant_to_dosare[s] for i in dosar_map[d]]

train_data = [converted[i] for i in train_idx]
test_data  = [converted[i] for i in test_idx]

print(f"Train: {len(train_data)}, Test: {len(test_data)}")
print(f"Overlap solicitanti: {len(set(train_sol) & set(test_sol))}")

train_dosare_set = set(d for s in train_sol for d in solicitant_to_dosare[s])
test_dosare_set  = set(d for s in test_sol  for d in solicitant_to_dosare[s])
print(f"Overlap dosare: {len(train_dosare_set & test_dosare_set)}")

In [ ]:
def make_dataset(data_list):
    return Dataset.from_dict({
        'input_ids':      [d['input_ids']      for d in data_list],
        'attention_mask': [d['attention_mask'] for d in data_list],
        'labels':         [d['labels']         for d in data_list],
    })

train_dataset = make_dataset(train_data)
test_dataset  = make_dataset(test_data)

model = AutoModelForTokenClassification.from_pretrained(
    'Davlan/xlm-roberta-base-ner-hrl',
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_labels, true_preds = [], []
    for pred, label in zip(predictions, labels):
        tl, tp = [], []
        for p, l in zip(pred, label):
            if l == -100: continue
            tl.append(id2label[l])
            tp.append(id2label[p])
        true_labels.append(tl)
        true_preds.append(tp)
    return {'f1': seq_f1(true_labels, true_preds, average='micro')}

training_args = TrainingArguments(
    output_dir='/kaggle/working/ner_model',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=20,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

print("Antrenare")
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=2)
ft_true, ft_pred = [], []
for pred, label in zip(preds, predictions.label_ids):
    tl, tp = [], []
    for p, l in zip(pred, label):
        if l == -100: continue
        tl.append(id2label[l])
        tp.append(id2label[p])
    ft_true.append(tl)
    ft_pred.append(tp)

ft_f1 = seq_f1(ft_true, ft_pred, average='micro')
print(seq_report(ft_true, ft_pred))
print(f"F1 micro: {ft_f1:.4f}")

In [ ]:
# baseline
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

baseline_tokenizer = AutoTokenizer.from_pretrained('Davlan/xlm-roberta-base-ner-hrl')
baseline_model_raw = AutoModelForTokenClassification.from_pretrained('Davlan/xlm-roberta-base-ner-hrl')

id2label_hrl = {
    0: 'O',
    1: 'B-PER', 2: 'I-PER',
    3: 'B-ORG', 4: 'I-ORG',
    5: 'B-LOC', 6: 'I-LOC',
    7: 'B-DATE', 8: 'I-DATE'
}
baseline_model_raw.config.id2label = id2label_hrl
baseline_model_raw.config.label2id = {v: k for k, v in id2label_hrl.items()}

baseline_ner = pipeline(
    "ner",
    model=baseline_model_raw,
    tokenizer=baseline_tokenizer,
    aggregation_strategy="simple",
    device=0
)


functionari = [
    'TRAIAN BASESCU', 'ADRIEAN VIDEANU', 'DUMITRU STANESCU',
    'LUDOVIC ORBAN', 'ADRIAN IORDACHE', 'CRISTINA SETRAN',
    'ANTON PETRISOR PARLAGI', 'RADU DINULESCU',
    'SORIN MIRCEA OPRESCU', 'SORIN OPRESCU',
]

def is_functionar(name):
    n = name.upper().strip()
    for f in functionari:
        if n in f or f in n:
            return True
    return False

test_tasks = [data[i] for i in test_idx]
all_true, all_pred_bl = [], []

for ti, task in enumerate(test_tasks):
    text = task['data']['text']

    true_anns = []
    anns = task.get('annotations', [])
    if anns and anns[0].get('result'):
        for r in anns[0]['result']:
            true_anns.append({
                'start': r['value']['start'],
                'end':   r['value']['end'],
                'label': r['value']['labels'][0]
            })

    pred_anns = []
    for offset in range(0, len(text), 450):
        chunk = text[offset:offset + 450]
        try:
            results = baseline_ner(chunk)
        except:
            continue
        for r in results:
            if r['score'] < 0.5:
                continue
            s, e = offset + r['start'], offset + r['end']
            entity_text = text[s:e]

            if r['entity_group'] in ('PER', 'ORG'):
                # acelasi filtru functionar ca la fine-tuned
                if is_functionar(entity_text):
                    continue
                # acelasi filtru footer ca la fine-tuned
                if s > len(text) * 0.8:
                    continue
                pred_anns.append({'start': s, 'end': e, 'label': 'PER_ANON'})

            elif r['entity_group'] == 'LOC':
                ctx = text[max(0, s-60):s].lower()
                if 'domiciliul' in ctx or 'domiciliat' in ctx:
                    pred_anns.append({'start': s, 'end': e, 'label': 'ADDR_DOM'})

    encoding = tokenizer(text, truncation=True, max_length=512,
                         return_offsets_mapping=True)
    offsets = encoding['offset_mapping']
    true_labels = ['O'] * len(offsets)
    pred_labels = ['O'] * len(offsets)

    for anns_list, lab_list in [(true_anns, true_labels), (pred_anns, pred_labels)]:
        for ann in anns_list:
            first = True
            for idx, (ts, te) in enumerate(offsets):
                if te == 0:
                    continue
                if (ts >= ann['start'] and te <= ann['end']) or \
                   (ts < ann['end'] and te > ann['start']):
                    lab_list[idx] = f"B-{ann['label']}" if first else f"I-{ann['label']}"
                    first = False

    all_true.append([l for l, (s, e) in zip(true_labels, offsets) if e > 0])
    all_pred_bl.append([l for l, (s, e) in zip(pred_labels, offsets) if e > 0])

    if (ti+1) % 20 == 0:
        print(f"  Baseline evaluat {ti+1}/{len(test_tasks)}")

bl_f1 = seq_f1(all_true, all_pred_bl, average='micro')
print(f"\nBASELINE (xlm-roberta zero-shot)")
print(seq_report(all_true, all_pred_bl))
print(f"F1 micro: {bl_f1:.4f}")

In [ ]:
print(f"BASELINE xlm-roberta zero-shot: F1 = {bl_f1:.4f}")
print(f"Fine-tuned:        F1 = {ft_f1:.4f}")
print(f"Improvement:                       +{ft_f1 - bl_f1:.4f}")

In [ ]:
from seqeval.metrics import classification_report
import plotly.graph_objects as go

ft_report = classification_report(ft_true, ft_pred, output_dict=True)
bl_report = classification_report(all_true, all_pred_bl, output_dict=True)

# PLOT 1
labels = ['ADDR_DOM', 'DOC_ID', 'PER_ANON']
ft_f1 = [ft_report[l]['f1-score'] for l in labels]
bl_f1 = [bl_report[l]['f1-score'] for l in labels]

fig1 = go.Figure(data=[
    go.Bar(name='XLM-RoBERTa zero-shot (baseline)', x=labels, y=bl_f1,
           marker_color='#ef553b', text=[f'{v:.2f}' for v in bl_f1],
           textposition='outside'),
    go.Bar(name='XLM-RoBERTa fine-tuned (PMB)', x=labels, y=ft_f1,
           marker_color='#636efa', text=[f'{v:.2f}' for v in ft_f1],
           textposition='outside'),
])
fig1.update_layout(
    title='F1 Score by Entity Type',
    yaxis=dict(title='F1 Score', range=[0, 1.15]),
    xaxis_title='Entity Type',
    barmode='group',
    template='plotly_white',
    legend=dict(orientation='h', y=-0.2)
)
fig1.write_html("/kaggle/working/plot_f1_per_entity.html")

#PLOT 2
metrics = ['Precision', 'Recall', 'F1']
bl_vals = [
    bl_report['micro avg']['precision'],
    bl_report['micro avg']['recall'],
    bl_report['micro avg']['f1-score']
]
ft_vals = [
    ft_report['micro avg']['precision'],
    ft_report['micro avg']['recall'],
    ft_report['micro avg']['f1-score']
]

fig2 = go.Figure(data=[
    go.Bar(name='XLM-RoBERTa zero-shot (baseline)', x=metrics, y=bl_vals,
           marker_color='#ef553b', text=[f'{v:.2f}' for v in bl_vals],
           textposition='outside'),
    go.Bar(name='XLM-RoBERTa fine-tuned (PMB)', x=metrics, y=ft_vals,
           marker_color='#636efa', text=[f'{v:.2f}' for v in ft_vals],
           textposition='outside'),
])
fig2.update_layout(
    title='Overall NER Performance (Micro Average)',
    yaxis=dict(title='Score', range=[0, 1.15]),
    xaxis_title='Metric',
    barmode='group',
    template='plotly_white',
    legend=dict(orientation='h', y=-0.2)
)
fig2.write_html("/kaggle/working/plot_overall.html")

print(f"\nFine-tuned  — Precision: {ft_vals[0]:.3f}, Recall: {ft_vals[1]:.3f}, F1: {ft_vals[2]:.3f}")
print(f"Zero-shot   — Precision: {bl_vals[0]:.3f}, Recall: {bl_vals[1]:.3f}, F1: {bl_vals[2]:.3f}")